# Stage 1 - Machine Learning Concepts

## Classical vehicle-price regression

This notebook is the foundational first stage of the project. It explores
anonymised AutoTrader adverts, prepares mixed numerical and categorical
features, and compares linear regression, K-Nearest Neighbours and a
decision tree.

The portfolio refactor fits learned preprocessing only on training data
and calculates instance-level errors from real predictions.


## 1. Setup

Install the project from the repository root with
`python -m pip install -e ".[notebooks,dev]"` and place `adverts.csv`
under `data/raw/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autotrader_price.data import load_vehicle_data, split_features_target
from autotrader_price.evaluation import evaluate_regressor
from autotrader_price.preprocessing import make_preprocessor

RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "adverts.csv"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)


## 2. Load and validate the data


In [ ]:
data = load_vehicle_data(DATA_PATH)
print(f"Rows after deterministic domain cleaning: {len(data):,}")
data.info()
data.head()


The loader derives missing registration years from valid registration
codes where possible, applies documented domain filters and removes the
listing identifier. Imputation, encoding and scaling are deferred until
after the hold-out split.


## 3. Exploratory data analysis


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(data=data, x="mileage", bins=50, ax=axes[0], color="seagreen")
axes[0].set_title("Mileage distribution")

sns.histplot(
    data=data,
    x="year_of_registration",
    bins=21,
    ax=axes[1],
    color="darkorange",
)
axes[1].set_title("Registration-year distribution")

top_makes = data["standard_make"].value_counts().head(10).index
sns.countplot(
    data=data[data["standard_make"].isin(top_makes)],
    y="standard_make",
    hue="standard_make",
    order=top_makes,
    legend=False,
    ax=axes[2],
    palette="crest",
)
axes[2].set_title("Ten most frequent makes")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

plot_sample = data.sample(min(30_000, len(data)), random_state=RANDOM_STATE)
sns.scatterplot(
    data=plot_sample,
    x="mileage",
    y="price",
    alpha=0.2,
    s=15,
    ax=axes[0],
)
axes[0].set_title("Mileage and advertised price")

yearly_price = data.groupby("year_of_registration", as_index=False)["price"].median()
sns.lineplot(
    data=yearly_price,
    x="year_of_registration",
    y="price",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Median advertised price by year")

top_makes = data["standard_make"].value_counts().head(10).index
sns.boxplot(
    data=plot_sample[plot_sample["standard_make"].isin(top_makes)],
    x="price",
    y="standard_make",
    order=top_makes,
    showfliers=False,
    ax=axes[2],
)
axes[2].set_title("Price by make (outliers hidden)")

plt.tight_layout()
plt.show()


These plots describe associations rather than causal effects. Advertised
price generally falls as mileage increases and rises for newer vehicles,
while make captures substantial market segmentation.


## 4. Train/test split and preprocessing


In [ ]:
X_train, X_test, y_train, y_test = split_features_target(
    data,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Training rows:", f"{len(X_train):,}")
print("Test rows:", f"{len(X_test):,}")


The preprocessing pipeline uses cross-fitted target encoding for
high-cardinality categoricals, one-hot encoding for low-cardinality
features, median or most-frequent imputation and a Yeo-Johnson
transformation for numerical variables.


## 5. Model selection


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

searches = {
    "Linear Regression": GridSearchCV(
        Pipeline(
            [
                ("preprocessor", make_preprocessor(scale_output=True)),
                ("model", LinearRegression()),
            ]
        ),
        {"model__fit_intercept": [True, False]},
        cv=5,
        scoring="r2",
        n_jobs=-1,
    ),
    "KNN": GridSearchCV(
        Pipeline(
            [
                ("preprocessor", make_preprocessor(scale_output=True)),
                ("model", KNeighborsRegressor(n_jobs=-1)),
            ]
        ),
        {
            "model__n_neighbors": [15, 25, 35, 50],
            "model__weights": ["uniform", "distance"],
        },
        cv=5,
        scoring="r2",
        n_jobs=-1,
    ),
    "Decision Tree": GridSearchCV(
        Pipeline(
            [
                ("preprocessor", make_preprocessor()),
                ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
            ]
        ),
        {
            "model__max_depth": [10, 20, 30],
            "model__min_samples_split": [2, 10, 20],
            "model__min_samples_leaf": [1, 5, 10],
        },
        cv=5,
        scoring="r2",
        n_jobs=-1,
    ),
}

for name, search in searches.items():
    print(f"Fitting {name}...")
    search.fit(X_train, y_train)
    print("Best parameters:", search.best_params_)


## 6. Hold-out evaluation


In [ ]:
rows = []
for name, search in searches.items():
    train_metrics = evaluate_regressor(search.best_estimator_, X_train, y_train)
    test_metrics = evaluate_regressor(search.best_estimator_, X_test, y_test)
    rows.append(
        {
            "model": name,
            "train_r2": train_metrics["r2"],
            "test_r2": test_metrics["r2"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
        }
    )

results = (
    pd.DataFrame(rows)
    .sort_values("test_r2", ascending=False)
    .reset_index(drop=True)
)
results


In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(16, 4.5), sharex=True, sharey=True)

for axis, row in zip(axes, results.itertuples()):
    model = searches[row.model].best_estimator_
    predictions = model.predict(X_test)
    axis.scatter(y_test, predictions, alpha=0.15, s=10)
    limits = [
        min(float(y_test.min()), float(predictions.min())),
        max(float(y_test.max()), float(predictions.max())),
    ]
    axis.plot(limits, limits, "k--", linewidth=1)
    axis.set_title(row.model)
    axis.set_xlabel("Actual price")

axes[0].set_ylabel("Predicted price")
plt.suptitle("Actual versus predicted prices")
plt.tight_layout()
plt.show()


## 7. Permutation importance


In [ ]:
from sklearn.inspection import permutation_importance

best_name = results.iloc[0]["model"]
best_model = searches[best_name].best_estimator_

importance = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="r2",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_series = pd.Series(
    importance.importances_mean,
    index=X_test.columns,
).sort_values()

importance_series.tail(8).plot(kind="barh", figsize=(8, 5), color="teal")
plt.title(f"Permutation importance - {best_name}")
plt.xlabel("Mean decrease in test R²")
plt.tight_layout()
plt.show()


## 8. Fine-grained errors from real predictions


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_positions = rng.choice(
    len(X_test),
    size=min(250, len(X_test)),
    replace=False,
)

error_frame = pd.DataFrame()
for name, search in searches.items():
    predictions = search.best_estimator_.predict(X_test.iloc[sample_positions])
    error_frame[name] = np.abs(
        y_test.iloc[sample_positions].to_numpy() - predictions
    )

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
error_frame.plot(ax=axes[0], alpha=0.75)
axes[0].set_title("Absolute error by sampled test observation")
axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("Absolute error")

sns.boxplot(data=error_frame, ax=axes[1])
axes[1].set_title("Distribution of real hold-out errors")
axes[1].set_ylabel("Absolute error")

plt.tight_layout()
plt.show()


## 9. Stage-one conclusion

The baseline study tests whether simple linear, distance-based and
tree-based models can explain vehicle-price variation. The original
submission recorded the strongest baseline result for the decision tree
(test R² 0.847). The next notebook extends this work with ensembles,
explainability and representation analysis.
